# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [5]:
FILE_A  = "inputs.csv"     # <- your main table's file name
TABLE_A = "inp"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "provision.csv"   # <- the related table's file name
TABLE_B = "prov"                 # <- a short name for it, e.g. "countries"

In [6]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/inputs.csv (0.1 MB)
Found data/provision.csv (0.2 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [88]:
con.sql(f"SELECT * FROM inp LIMIT 10")

┌──────────┬──────────────────────────┬─────────────────┬────────────┬──────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬────────────────────────────────────────────┬────────────────────────────┬─────────────

In [89]:
con.sql(f"SELECT * FROM prov LIMIT 10")

┌───────────────┬─────────────┬────────────────────────────────────────────┬─────────────┬────────────────┬───────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ provider_name │ provider_id │               provided_name                │ provided_id │ share_provided │ year  │                                                      source                                                       │
│    varchar    │   varchar   │                  varchar                   │   varchar   │     double     │ int64 │                                                      varchar                                                      │
├───────────────┼─────────────┼────────────────────────────────────────────┼─────────────┼────────────────┼───────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ ACM Research  │ P313        │ Etch and clean tools                    

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [90]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

inp: 126 rows loaded
prov: 1,300 rows loaded


In [91]:
con.sql(f"DESCRIBE {TABLE_A}")

┌────────────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│                column_name                 │ column_type │  null   │   key   │ default │  extra  │
│                  varchar                   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ input_id                                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ input_name                                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type                                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stage_name                                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stage_id                                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description                                │ VARCHAR     │ YES     │ NULL    │ NULL    │ 

In [92]:
con.sql(f"DESCRIBE {TABLE_B}")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ provider_name  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ provider_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ provided_name  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ provided_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ share_provided │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ year           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ source         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [93]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌────────────────────────────────────────────┬─────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────┬────────────────────┬────────────────────┬─────────┬─────────┬─────────┬───────┬─────────────────┐
│                column_name                 │ column_type │                                                        min                                                        │              

In [98]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌────────────────┬─────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬───────┬─────────────────┐
│  column_name   │ column_type │                                                        min                                                        │                                                                     max                                                                     │ approx_unique │        avg        │        std         │        q25         │        q50        │        q75         │ count │ null_percentage │
│    varchar     │   varchar   │                                                      varchar                                   

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [12]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv('{DATA_A}', nullstr = ['NA', 'N/A', '','NULL'])
""")
con.sql(f"DESCRIBE {TABLE_A}")

┌────────────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│                column_name                 │ column_type │  null   │   key   │ default │  extra  │
│                  varchar                   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ input_id                                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ input_name                                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type                                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stage_name                                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stage_id                                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description                                │ VARCHAR     │ YES     │ NULL    │ NULL    │ 

In [69]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"CREATE OR REPLACE TABLE provision AS SELECT * FROM read_csv('{DATA_B}')")

con.sql(f"""
    CREATE OR REPLACE TABLE provision AS
    SELECT * REPLACE (COALESCE(share_provided, 0) AS share_provided)
    FROM read_csv('{DATA_B}', nullstr = ['NA', 'N/A', '', 'NULL'])
""")
con.sql(f"DESCRIBE provision")
con.sql(f"SELECT COUNT(*) FROM provision WHERE share_provided IS NULL")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

**What did you fix, and how did you know?** DATA_B had rows where there is no value in the column share_provided, I am replacing those cells with 0 so that now no rows should have a NULL value for share_provided, also naming the new table provision



## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [53]:
KEY_A = "input_id"   # <- the shared column's name in your main table
KEY_B = "provided_id"   # <- its name in the second table

In [95]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM inp),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM provisions)
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│       126 │       113 │          110 │
└───────────┴───────────┴──────────────┘

**Describe the relationship** (double-click to edit):

- **Key column:** in the main table it's called input_id, in the second provided_id
- **How they relate:** does one row in the second table match *one* row in the main table, or *many*? one row in the first table may match many in the second because there are multiple providers for the same input in chip design
- **How well they match:** how many keys appear in only one table? Give one example and a guess at why (spelling, codes vs. names, regions mixed in with countries...) 16 keys only appear in the first table, perhaps some steps of the chip design process are so critical that their providers dont disclose info



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [96]:
con.sql("""SELECT 
        count(*) AS row_count
    FROM inp""")



┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│       126 │
└───────────┘

In [97]:
con.sql("""SELECT 
        count(*) AS row_count
    FROM prov""")



┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│      1300 │
└───────────┘

In [84]:
con.sql(f"CREATE OR REPLACE TABLE inputs AS SELECT * FROM read_csv('{DATA_A}')")
con.sql("""SELECT 
        count(input_id) AS input_count, 
        min(year) AS first_year,
        max(year) AS last_year,
    FROM inputs""")


┌─────────────┬────────────┬───────────┐
│ input_count │ first_year │ last_year │
│    int64    │   int64    │   int64   │
├─────────────┼────────────┼───────────┤
│         126 │       2019 │      2025 │
└─────────────┴────────────┴───────────┘

In [83]:
con.sql("""SELECT 
        min(year) AS first_year,
        max(year) AS last_year,
    FROM provisions""")


┌────────────┬───────────┐
│ first_year │ last_year │
│   int64    │   int64   │
├────────────┼───────────┤
│       2019 │      2025 │
└────────────┴───────────┘

**What this tells us:** There are 126 unique components involved in chip design, data documented from 2019 to 2025

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [64]:
con.sql(f"CREATE OR REPLACE TABLE provisions AS SELECT * FROM read_csv('{DATA_B}')")

con.sql("""SELECT count(DISTINCT provider_name) AS n_providers
    FROM provisions
    WHERE share_provided > 50""")

┌─────────────┐
│ n_providers │
│    int64    │
├─────────────┤
│          33 │
└─────────────┘

**What this tells us:** There are 33 providers where they have a more than half market share of a procedure in chip design

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [71]:
con.sql("""
    SELECT provider_name, avg(share_provided) AS avg_share FROM provision
    GROUP BY provider_name
""")

┌───────────────────┬────────────────────┐
│   provider_name   │     avg_share      │
│      varchar      │       double       │
├───────────────────┼────────────────────┤
│ ACM Research      │               4.58 │
│ ASM International │ 13.062500000000002 │
│ ASTI              │                5.1 │
│ AUT               │            24.9375 │
│ Advantest         │ 39.480000000000004 │
│ Applied Materials │  40.70322580645162 │
│ BLR               │ 0.8666666666666667 │
│ Bruker            │                4.9 │
│ CAMTEK            │              5.625 │
│ CETC              │                2.2 │
│  ·                │                 ·  │
│  ·                │                 ·  │
│  ·                │                 ·  │
│ Sumitomo          │                0.0 │
│ TNSC              │                0.0 │
│ TSMC              │                0.0 │
│ Tanaka            │                0.0 │
│ Thomas West       │                0.0 │
│ Tokyo Ohka Kogyo  │                0.0 │
│ Xpeedic  

**What this tells us:** For every provider, averaging their share provided across each step they are involved with in chip design.

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [73]:
con.sql("""
    SELECT provider_name, count(DISTINCT provided_id) AS n_inputs FROM provision
    GROUP BY provider_name
    ORDER BY n_inputs DESC LIMIT 3
""")

┌───────────────┬──────────┐
│ provider_name │ n_inputs │
│    varchar    │  int64   │
├───────────────┼──────────┤
│ JPN           │       93 │
│ USA           │       84 │
│ CHN           │       79 │
└───────────────┴──────────┘

**What this tells us:** Among all the providers involved in chip design, the top 3 entites that provide the most parts of a chip are Japan, USA, and China

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [ ]:
con.sql(f"""
    SELECT count(*) - count(share_provided) FROM provisions
""")

┌────────────────────────────────────────┐
│ (count_star() - count(share_provided)) │
│                 int64                  │
├────────────────────────────────────────┤
│                                    246 │
└────────────────────────────────────────┘

In [80]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│       126 │       113 │          110 │
└───────────┴───────────┴──────────────┘

**What this tells us:** In the original provisions data, there were 246 rows without data for market share.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. There are 246 rows that return share provided as zero which feels unrealistic, likely missing data
2. There are three steps in chip design not represented in the input data even though they were recorded in its provisions
3. There are sixteen steps in chip design not represented in the provisions data

**One question you'd want a non-technical person to be able to ask of this data:**



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** | Semiconductor Supply Chain Dataset - inputs | Semiconductor Supply Chain Dataset - provisions |
| **Source (link)** | https://github.com/georgetown-cset/eto-chip-explorer/tree/main/data | https://github.com/georgetown-cset/eto-chip-explorer/tree/main/data |
| **Who collected it, and why** | CSET analysts | CSET analysts |
| **License or terms of use** | https://eto.tech/dataset-docs/chipexplorer/ | https://eto.tech/dataset-docs/chipexplorer/ |
| **Rows × columns** | 126x10 | 1300x7 |
| **Time period covered** | 2019-2025 | 2019-2025 |
| **How often it's updated** | yearly | yearly |

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [99]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
